In [520]:
import numpy as np
import random

In [522]:
# Loss Functions / Reward Functions

# Circle of fifths distance
'''
    Assumes 0<=n<=12
'''
def circ_dist(n: int) -> int:
    if (n%2)==0: return min(n, abs(n-12))
    else: return abs(min(n, abs(n-12))-6)

'''
  - X : X (4,n)
  - C : List of chords in their mod12 values (m,n)
'''
def chord_similarity(X: np.ndarray, C: np.ndarray):
    # Take min circle of fifths distance from chord for each note
    n = X.shape[1]
    m = C.shape[0]
    X = np.repeat(X.T.flatten(), m, axis=0)
    C = np.repeat(C, 4, axis=1)
    Y = X-C
    Y = Y %12
    circ_distv = np.vectorize(circ_dist)
    Y = circ_distv(Y)
    score = np.min(Y, axis=0)
    score = np.matrix(score)
    score = score.reshape((4,n))
    return score.T * -1

'''
    Assumes X.shape = (4,n)
'''
def get_distances(X: np.ndarray) -> np.ndarray:
    DIST_MATRIX = np.matrix([
        [1,-1,0,0],
        [1,0,-1,0],
        [1,0,0,-1],
        [0,1,-1,0],
        [0,1,0,-1],
        [0,0,1,-1]
    ])
    return DIST_MATRIX * X


def interval_dist(x: int) -> int:
    return 12 if (x != 0 and x%12==0) else x%12
interval_distv = np.vectorize(interval_dist)

'''
    Assumes X.shape = (4,n)
'''
def get_interval_distances(X: np.ndarray) -> np.ndarray:
    return interval_distv(get_distances(X))  #(6,n) matrix

'''
    Assumes X.shape = (4,n)
'''
def parallel_octaves(X: np.ndarray) -> np.ndarray:
    n = X.shape[1]
    DISTS = get_interval_distances(X) #(6,n) matrix
    is12 = np.vectorize(lambda x : 1 if x==12 else 0)
    DISTS = is12(DISTS)
    array = []
    for i in range(n-1):
        row = [0]*n
        row[i] = 1
        row[i+1] = 1
        array.append(row)
    MOVES = np.array(array).T
    return np.vectorize(lambda x : 1 if x==2 else 0)(DISTS * MOVES) * -1

'''
    Assumes X.shape = (4,n)
'''
def parallel_fifths(X: np.ndarray) -> np.ndarray:
    n = X.shape[1]
    DISTS = get_interval_distances(X) #(6,n) matrix
    is7 = np.vectorize(lambda x : 1 if x==7 else 0)
    DISTS = is7(DISTS)
    array = []
    for i in range(n-1):
        row = [0]*n
        row[i] = 1
        row[i+1] = 1
        array.append(row)
    MOVES = np.array(array).T
    return np.vectorize(lambda x : 1 if x==2 else 0)(DISTS * MOVES) * -1

    
'''
    Takes in output from parallel_octaves or parallel_fifths: (6,n-1) matrix
    Returns (4,n) matrix
'''
def moves_to_indiv(MOVES: np.ndarray) -> np.ndarray:
    MOVESL = np.hstack((MOVES, ([[0]]*6)))
    MOVESR = np.hstack(([[0]]*6, MOVES))
    MOVES = MOVESL + MOVESR
    SUM_MATRIX = np.matrix([
        [1,1,1,0,0,0], #represents how we differenced in get_distances
        [1,0,0,1,1,0],
        [0,1,0,1,0,1],
        [0,0,1,0,1,1]
    ])
    return (SUM_MATRIX *MOVES)

'''
    Assumes X.shape = (4,n)
    Penalizes if higher voice is lower than lower voice
'''
def crossings(X: np.ndarray) -> np.ndarray:
    DISTS = get_distances(X)
    DISTS = np.vectorize(lambda x : 1 if x > 0 else 10*x)(DISTS)
    SUM_MATRIX = np.matrix([
        [1,1,1,0,0,0], #represents how we differenced in get_distances
        [1,0,0,1,1,0],
        [0,1,0,1,0,1],
        [0,0,1,0,1,1]
    ])
    RESULT = (SUM_MATRIX *DISTS)
    return RESULT - np.full(RESULT.shape, 3)

'''
    Assumes X.shape = (4,n)
'''
def crossing_info(X: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    DISTS = get_distances(X)
    DISTS = np.vectorize(lambda x : 0 if x > 0 else (x-1))(DISTS)
    ARR1 = []
    ARR2 = []
    ARR3 = []
    ARR1.append(DISTS[0, :] * (-1))
    ARR1.append(DISTS[0, :])
    ARR1.append(DISTS[1, :])
    ARR1.append(DISTS[2, :])
    ARR2.append(DISTS[1, :]*(-1))
    ARR2.append(DISTS[3, :]*(-1))
    ARR2.append(DISTS[3, :])
    ARR2.append(DISTS[4, :])
    ARR3.append(DISTS[2, :]*(-1))
    ARR3.append(DISTS[4, :]*(-1))
    ARR3.append(DISTS[4, :]*(-1))
    ARR3.append(DISTS[5, :])
    return np.vstack(ARR1), np.vstack(ARR2), np.vstack(ARR3)


'''
    Assumes X.shape = (4,n)
'''
def continuity(X: np.ndarray) -> np.ndarray:
    n = X.shape[1]
    array = []
    for i in range(n-1):
        row = [0]*n
        row[i] = 1
        row[i+1] = -1
        array.append(row)
    MOVES = np.array(array).T
    MOVES = X * MOVES
    return MOVES
'''
    Assumes X.shape = (4,n)
'''
def continuityR(X: np.ndarray) -> np.ndarray:
    MOVES = continuity(X)
    return np.hstack((MOVES, ([[0]]*4))) * -1
'''
    Assumes X.shape = (4,n)
'''
def continuityL(X: np.ndarray) -> np.ndarray:
    MOVES = continuity(X)
    return np.hstack(([[0]]*4, MOVES))


'''
    Assumes X.shape = (4,n)
'''
def update(X: np.ndarray) -> np.ndarray:
    CHORD_SIM = chord_similarity(X,C) # vals from 0 to -6
    PAR_OCT = moves_to_indiv(parallel_octaves(X)) # vals from 0 to 1
    PAR_FIF = moves_to_indiv(parallel_fifths(X)) # vals from 0 to 1
    CROSS_INFO = crossing_info(X) # 0 (if no need to move) otherwise -88 to 88
    CONTINUR = continuityR(X) # -88 to 88 (direction of right in relation to node)
    CONTINUL = continuityL(X) # -88 to 88 (direction of left in relation to node)
    # for each index in X
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            curr_pos = X[i,j]
            usd = [0.01, 1, 0.01] #up/stay/down
            # Let chord sim affect probabilities
            chord_sim_strength = 0.2
            usd[0] += (CHORD_SIM[i,j]**2) * chord_sim_strength
            usd[2] += (CHORD_SIM[i,j]**2) * chord_sim_strength
            # Let par_oct affect probabilities
            par_oct_strength = 40
            usd[0] += PAR_OCT[i,j] * par_oct_strength
            usd[2] += PAR_OCT[i,j] * par_oct_strength
            # Let par_fif affect probabilities
            par_fif_strength = 40
            usd[0] += PAR_FIF[i,j] * par_fif_strength
            usd[2] += PAR_FIF[i,j] * par_fif_strength
            # Let crossing info affect probabilites
            cross_info_strength = 10
            for CROSS_INFOI in CROSS_INFO:
                cross = CROSS_INFOI[i,j]
                if (cross > 0):
                    usd[0] += cross * cross_info_strength
                elif (cross < 0):
                    usd[2] += abs(cross * cross_info_strength)
            # Let continuity affect probabilities
            continu_strength = 0.01
            right = CONTINUR[i,j]
            left = CONTINUL[i,j]
            if (right > 0):
                usd[0] += right * continu_strength
            elif (right < 0):
                usd[2] += abs(right * continu_strength)
            if (left > 0):
                usd[0] += right * continu_strength
            elif (left < 0):
                usd[2] += abs(right * continu_strength)
            # Go up/stay/down based on usd probability
            total = 0
            for num in usd:
                total += num
            prob = random.random() * total
            run_sum = 0
            for dir in range(len(usd)):
                run_sum += usd[dir]
                if (run_sum >= prob):
                    X[i,j] += (1-dir) # If 0: +1, if 1, +0, if 2: -1
                    if (X[i,j] > 88):
                        X[i,j] = 88
                    if (X[i,j] < 0):
                        X[i,j] = 0
                    break
    return X

# Chords: I IV V 1 (A Major)
C = np.matrix([
  [0,4,7],
  [5,9,0],
  [7,11,2],
  [0,4,7]
])
C = C.T
X = np.matrix([[random.randint(0,88) for j in range(4)] for i in range(4)])
print(X)


for _ in range(1_000):
    X = update(X)

print(X.T)
score = chord_similarity(X, C)
print(score)
print(parallel_octaves(X))
print(moves_to_indiv(parallel_octaves(X)))
print(parallel_fifths(X))
print(moves_to_indiv(parallel_fifths(X)))
print(crossings(X))
print(continuityR(X))
print(continuityL(X))
print(crossing_info(X))

[[42 14 43 53]
 [21 84 14 84]
 [30 51 17 20]
 [40 88 87 46]]
[[81 78 40 31]
 [88 84 40 34]
 [87 85 43 40]
 [88 86 46 37]]
[[-1 -1 -4  0]
 [-2  0 -2 -1]
 [ 0 -1  0 -2]
 [ 0 -1 -1 -3]]
[[0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]]
[[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]
[[0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]
 [0 0 0]]
[[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]
[[0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]
 [0 0 0 0]]
[[ 7 -1  1  0]
 [ 6  1  1  0]
 [ 0  3  3  0]
 [ 3  6 -3  0]]
[[ 0 -7  1 -1]
 [ 0 -6 -1 -1]
 [ 0  0 -3 -3]
 [ 0 -3 -6  3]]
(matrix([[0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0]]), matrix([[0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0]]), matrix([[0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 0]]))
